# LangGraph - Components

In this chapter, we will examine some of the fundamental components of LangGraph.

* State

* Node
* Edge
* Graph
* StateGraph
* Tool
* Runnable
* Message
* Reducer

## State

State is the shared data structure that holds the entire "memory" or snapshot of your workflow at any given moment.

It's tipcally a `TypedDict` or a Pydantic model that defines all fields you want to track (e.g., conversation messages, counters, tool outputs).

In [1]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage
from langgraph.graph import add_messages

class State(TypedDict):
    count: int
    messages: Annotated[List[BaseMessage], add_messages]

State enables workflows to carry context (like conversation or counters) across multiple cycles.

## Node

A *node* is essentially a *unit of computation* — tipically a Python function — that takes in the current `state` (and optionally a `config`) and returns a partial update to that state.

![[source](https://cobusgreyling.medium.com/langgraph-from-langchain-explained-in-simple-terms-f7cd0c12cdbf)](assets/img/04-nodes-1.webp)

It can be synchronous or asynchronous.

In [2]:
from langchain_core.runnables import RunnableConfig

def my_node(state: State, config: RunnableConfig = None) -> dict:
    # Do something with the state and config
    new_message = BaseMessage(content="Hello, World!")
    return {"messages": [new_message]}

It returns a dict of state updates, not a full state, and the framework then merges those updaes using reducers (later o this subject).

## Edge

An *edge* defines the flow of execution — how state is passed betwenn nodes, and when the graph begins and ends.

![[source](https://www.getzep.com/ai-agents/langchain-agents-langgraph)](assets/img/04-edges-1.webp)

An edge connects two nodes (or special started/end points) and determines how execution moves after a node finishes processing.

### Normal Edges

Straightforward links from one node to another

In [3]:
from langgraph.graph import StateGraph
graph = StateGraph(State)

# adding a straightforward edge between two nodes
graph.add_edge("node_a", "node_b")

![](assets/img/04-edges-2.webp)

State simply flows from `node_a` to `node_b` next.

### Conditional Edges

Control flow depending on the current `state`. They use a function to decide which nodes to trigger next:

In [4]:
def routing_fn(state: State) -> bool:
    return state["count"] < 5

graph.add_conditional_edges(
    "node_a",
    routing_fn,
    {True: "node_b", False: "node_c"}
)

![](assets/img/04-edges-3.webp)

The `routing_fn` may return a node name (or list) or send custom states states via `Send` objects

### START & END Edges

Designate entry and termination points:

In [5]:

from langgraph.graph import START, END

graph.add_edge(START, "node_a") # Entry node
graph.add_edge("node_z", END)   # Flow ends here

![](assets/img/04-edges-4.webp)

You can also define conditional choices at entry using

```python
add_contitional_edges(START, ...)
```

## Graph

A *graph* is the orchestrator of our workflow. It combines the three core components — State, Nodes, and Edges — into a structured, executable system.

We define nodes, edges, and state, compile everything, and run it — LangGraph ensures correctness, flow control, and reliability.

Graphs are designed for looping, branching parallel, and even dynamic workflow patters, making them excellent for agentic applications.

## StateGraph

The `StateGraph` class in LangGraph is the primary way to build stateful, graph-based workflows.

It is a **graph builder** we intialize by passing our state schema to define the structure of the shared state across our workflow.

Internally, each node returns a **partial state update**, and `StateBraph` automatomatically merges these updates using reducers defined per state key.

In [6]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

class Point(TypedDict):
    x: int
    y: int

def move_point(state: Point):
    print(f"Current point: ({state['x']}, {state['y']})")
    return {"x": state["x"] + 1, "y": state["y"] + 1}

def should_continue(state: Point):
    return "move_point" if state["x"] < 10 else "END"

graph = StateGraph(Point)
graph.add_node("move_point", move_point)
# Set entry
graph.add_edge(START, "move_point")
# Conditional branching to itself or end
graph.add_conditional_edges("move_point", should_continue, {"move_point": "move_point", "END": END})
app = graph.compile()
result = app.invoke({"x": 0, "y": 0})
print(result)

Current point: (0, 0)
Current point: (1, 1)
Current point: (2, 2)
Current point: (3, 3)
Current point: (4, 4)
Current point: (5, 5)
Current point: (6, 6)
Current point: (7, 7)
Current point: (8, 8)
Current point: (9, 9)
{'x': 10, 'y': 10}


Why Use `StateGraph`?

* Strikes a balance between typed structure and graph-based flexibility.

* Handles merging, routing, and execution lifecycle so you don't have to.
* Well-suited for agent workflows, tool calls, and dynamic decision loops — more structured than bare Graph.

## Tool

The *Tool* component (often implemented via `ToolNode`) provides a structured way for graphs to integrate and orchestrate calls to external functionality, like APIs, database queries, calculators, or custom helper functions.

![[text](https://x.com/LangChainAI/status/1799109018163761588)](assets/img/04-tools-1.webp)

A Tool is a callable external function wrapped with metadata (name, description, args schema) using LangChain's `@tool` decorator.

![https://langchain-ai.github.io/langgraph/concepts/tools/](assets/img/04-tools-2.webp)

It becomes part of the graph by being wrapped into a *ToolNode*, a specialized `Runnable` that handles the execution lifecycle when the graph invokes a tool call.

We can bind tools to an LLM using `.bind_tools([...])`, enabling the model to emit tool calls during generation.

When the LLM outputs a structured tool call (`tool_calls`), the graph routes that to a `ToolNode`.

`ToolNode` takes the tool name and args from the call, finds and invokes the corresponding function in the runtime environment.

It wraps results (or exceptions) into a `ToolMessage` allowing the graph to handle failures gracefully or propagate errors if configured.

## Runnable

In LangGraph (and LangChaini more broadly), *Runnables* are the modular, composable units of computation and workflow orchestration.

Think of them as function-like building blocs that can be chained, batched, streamed, and reused.

## Message

A message is a unit of communication used primarily in chat models and workflows.

`messages` refers to a special state channel used to track a conversation as a list of chat-like messages.

LangChain uses `BaseMessage` subclasses: `HumanMessage`, `AIMessage`, `SystemMessage`, `ToolMessage`, etc. (or serialized JSON forms).

Messages are commonly stored as a list under the `messages` state key.

In [7]:
class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

These are the primary message classes used in conversation flows:

|Message Type| Role & Purpose|
|---|---|
|`SystemMessage`|Provides system-level instructions or context (e.g., persona, rules)|
|`HumanMessage`|Represents input from a human user|
|`AIMessage`|Output from the AI/chat model, may include tool call info|
|`ToolMessage`|Encapsulates results returned by a tool invocation|
|`FunctionMessage`|*Legacy*: used when using OpenAI's function-calling API|

Streaming/Chunked Message classes support streaming and are often used in token-by-token streaming contexts:

`AIMessageChunk`, `HumanMessageChunk`, `SystemMessageChunk`, `ToolMessageChunk`, `FunctionMessageChunk`

They represent partial/chunked streams of the corresponding message types.

## Reducer

A reducer is a function that determines **how state updates are combined** when multiple nodes in a graph update the same state key. Reducers define whether updates are **overwritten** or **merged/accumulated**.

**Without a reducer**: any node update replaces the existing value.

* Example: node returns `{"foo":2}` overwites previous `foo`.

**With a reducer**: updates are merged via a function, combining old and new values.

* Example: defining `bar: Annotated[List[str], add]` causes new lists to be concatenated — `["a"] + ["b"] = ["a","b"]`.

When nodes run concurrently and update the same key, LangGraph needs a reducer to resolve how to combine them; otherwise, it throws `InvalidConcurrentGraphUpdate`.

In [8]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph.message import add_messages

class State(TypedDict):
    numbers: Annotated[List[int], add]
    messages: Annotated[List[BaseMessage], add_messages]